# FleetLogix — Avance 2: SQL Analysis y optimización

**Proyecto Integrador | Henry — Módulo 2**  
**Autor:** Luis Fernando Cornejo Martínez  
**Motor:** PostgreSQL · **Base:** `fleetlogix` · **Esquema:** `public`

## Objetivo
Ejecutar y documentar las **12 consultas SQL originales** proporcionadas por Henry (3 básicas, 5 intermedias y 4 complejas), explicar su utilidad de negocio, examinar planes con `EXPLAIN (ANALYZE, BUFFERS)`, diseñar **5 índices** a partir de la evidencia y comparar tiempos **antes/después**.

**Reglas de trabajo:** no volver a ejecutar el generador de datos; conservar el SQL original; no interpretar resultados vacíos como errores sin revisar las fechas; no crear índices hasta completar la línea base. Este notebook no ejecuta cambios DDL automáticamente.

## 1. Preparación y conexión segura

Antes de iniciar el kernel, configura `FLEETLOGIX_PASSWORD` en el entorno de ejecución. El notebook **no almacena ni imprime la contraseña**. Si no está configurada, se solicita de manera oculta durante la sesión. Se utiliza una conexión de **solo lectura** para las consultas y las mediciones. Para crear índices se proporciona más adelante un script SQL separado que debe revisarse y ejecutarse manualmente en DBeaver.

Dependencias: `psycopg2-binary`, `pandas`, `IPython` (normalmente ya instaladas en tu entorno virtual).

In [1]:
import os
import getpass
import time
import statistics
from datetime import datetime
from pathlib import Path
import pandas as pd
import psycopg2
from IPython.display import display

DB_CONFIG = {
    "host": os.getenv("FLEETLOGIX_HOST", "localhost"),
    "port": int(os.getenv("FLEETLOGIX_PORT", "5432")),
    "dbname": os.getenv("FLEETLOGIX_DB", "fleetlogix"),
    "user": os.getenv("FLEETLOGIX_USER", "postgres"),
    "password": os.getenv("FLEETLOGIX_PASSWORD") or getpass.getpass("Contraseña PostgreSQL: "),
    "options": "-c search_path=public",
}
conn = psycopg2.connect(**DB_CONFIG)
conn.set_session(readonly=True, autocommit=True)
print("Conexión de solo lectura establecida.")
print("Fecha de ejecución:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Base de datos:", DB_CONFIG["dbname"])


Conexión de solo lectura establecida.
Fecha de ejecución: 2026-11-13 17:34:53
Base de datos: fleetlogix


### Funciones auxiliares de ejecución y medición

Se emplea el cursor de `psycopg2` directamente. `EXPLAIN ANALYZE` **ejecuta** cada SELECT, por lo que conviene usarlo solo con consultas de lectura. El tiempo reportado por PostgreSQL puede diferir del tiempo medido desde Python por la transferencia de resultados y la carga del sistema.

Cada medición se repite tres veces; guardamos la **mediana del `Execution Time` de PostgreSQL** para reducir fluctuaciones. No se ejecuta ningún índice aquí.

In [2]:
def consultar(sql, limite_visualizacion=15):
    with conn.cursor() as cur:
        cur.execute(sql)
        columnas = [d[0] for d in cur.description]
        filas = cur.fetchall()
    resultado = pd.DataFrame(filas, columns=columnas)
    display(resultado.head(limite_visualizacion))
    print(f"Filas devueltas: {len(resultado):,}")
    return resultado


def explicar(sql):
    with conn.cursor() as cur:
        cur.execute("EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) " + sql.strip().rstrip(";"))
        lineas = [fila[0] for fila in cur.fetchall()]
    plan = "\n".join(lineas)
    print(plan)
    return plan


def medir(sql, repeticiones=3):
    import re
    tiempos = []
    for _ in range(repeticiones):
        with conn.cursor() as cur:
            cur.execute("EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) " + sql.strip().rstrip(";"))
            plan = "\n".join(fila[0] for fila in cur.fetchall())
        coincidencia = re.search(r"Execution Time:\s*([0-9.]+)\s*ms", plan)
        if not coincidencia:
            raise RuntimeError("No se pudo extraer Execution Time del plan")
        tiempos.append(float(coincidencia.group(1)))
    return round(statistics.median(tiempos), 3)

consultas = {}
resultados = {}
planes_antes = {}
planes_despues = {}
tiempos_antes = {}
tiempos_despues = {}


## 2. Comprobaciones iniciales

Los datos son sintéticos. Las consultas con ventanas basadas en `CURRENT_DATE` pueden quedar vacías si los viajes y las entregas no cubren las fechas recientes. Este diagnóstico **no altera** las consultas oficiales.

In [3]:
sql_diagnostico = """
SELECT 'trips' AS tabla, COUNT(*) AS registros,
       MIN(departure_datetime) AS fecha_minima, MAX(departure_datetime) AS fecha_maxima
FROM public.trips
UNION ALL
SELECT 'deliveries', COUNT(*), MIN(scheduled_datetime), MAX(scheduled_datetime)
FROM public.deliveries;
"""
display(consultar(sql_diagnostico))


,tabla,registros,fecha_minima,fecha_maxima
0,trips,100000,2024-11-05 01:22:03.178014,2026-11-05 23:11:14.218014
1,deliveries,400000,2024-11-05 02:38:21.498618,2026-11-06 00:56:14.218014


Filas devueltas: 2


,tabla,registros,fecha_minima,fecha_maxima
0,trips,100000,2024-11-05 01:22:03.178014,2026-11-05 23:11:14.218014
1,deliveries,400000,2024-11-05 02:38:21.498618,2026-11-06 00:56:14.218014


**Nota sobre consultas con fechas relativas:** las consultas **2, 4, 6, 8, 10 y 12** dependen de la fecha actual. Si no devuelven filas, documenta la cobertura temporal de los datos. Puedes crear variantes exploratorias con fechas fijas, pero no sustituyas silenciosamente las consultas originales de Henry.

**Nota de calidad SQL:** la consulta 9 une `trips` y `maintenance` por vehículo en una misma agregación; esto puede duplicar kilómetros y costos. Conservamos el SQL oficial para el análisis inicial y dejamos una sección para documentar una alternativa corregida.

## 3. Consultas básicas (1–3)

### Query 1. Contar vehículos por tipo

**Problema de negocio:** Conocer la composición de la flota.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Análisis pendiente de completar con resultados reales:**
-**Hallazgo de negocio:** FleetLogix dispone de 200 vehículos: 69 vans (34.5 %), 60 camiones grandes (30 %), 51 camiones medianos (25.5 %) y 20 motocicletas (10 %). Las vans y los camiones grandes representan conjuntamente el 64.5 % de la flota.

**Operadores del plan relevantes:** PostgreSQL utiliza `Seq Scan` para recorrer los 200 registros de `vehicles`, `HashAggregate` para agruparlos por tipo y `Sort` mediante quicksort para ordenar los resultados.

**Rendimiento inicial:** El tiempo base registrado mediante tres mediciones y su mediana fue de **0.134 ms**. La ejecución individual mostrada por `EXPLAIN ANALYZE` fue de **0.184 ms**, con un tiempo de planificación de **0.115 ms**.

**Observaciones y limitaciones:** La consulta devuelve correctamente los cuatro tipos de vehículos y su suma coincide con los 200 registros esperados. El recorrido secuencial resulta apropiado para el tamaño actual de la tabla. No se identifica una necesidad inmediata de optimización mediante índices. Los tiempos pueden variar según la carga del sistema y el estado de la caché.


In [4]:
# Query 1 — SQL original del archivo 02_queries_analysis.sql
consultas[1] = 'SELECT \n    vehicle_type,\n    COUNT(*) as cantidad\nFROM vehicles\nGROUP BY vehicle_type\nORDER BY cantidad DESC;'
print(consultas[1])


SELECT 
    vehicle_type,
    COUNT(*) as cantidad
FROM vehicles
GROUP BY vehicle_type
ORDER BY cantidad DESC;


In [5]:
# Ejecutar esta celda ANTES de crear los índices
resultados[1] = consultar(consultas[1])
planes_antes[1] = explicar(consultas[1])
tiempos_antes[1] = medir(consultas[1])
print("Tiempo base (mediana, ms):", tiempos_antes[1])


,vehicle_type,cantidad
0,Van,69
1,Camión Grande,60
2,Camión Mediano,51
3,Motocicleta,20


Filas devueltas: 4
Sort  (cost=5.08..5.09 rows=4 width=19) (actual time=0.137..0.137 rows=4.00 loops=1)
  Sort Key: (count(*)) DESC
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=2
  ->  HashAggregate  (cost=5.00..5.04 rows=4 width=19) (actual time=0.128..0.130 rows=4.00 loops=1)
        Group Key: vehicle_type
        Batches: 1  Memory Usage: 32kB
        Buffers: shared hit=2
        ->  Seq Scan on vehicles  (cost=0.00..4.00 rows=200 width=11) (actual time=0.023..0.038 rows=200.00 loops=1)
              Buffers: shared hit=2
Planning Time: 0.115 ms
Execution Time: 0.184 ms
Tiempo base (mediana, ms): 0.134


### Query 2. Listar conductores con licencia próxima a vencer

**Problema de negocio:** Prevenir problemas legales por licencias vencidas.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.


**Hallazgo de negocio:** La consulta identificó 9 conductores cuyas licencias vencen antes del límite de 30 días establecido. Esta información permite anticipar renovaciones y reducir riesgos operativos y legales.

**Plan de ejecución:** PostgreSQL utilizó `Seq Scan` sobre la tabla `drivers`, evaluando 400 registros. El filtro descartó 391 y devolvió 9. Posteriormente aplicó un ordenamiento `Sort` mediante `quicksort`, utilizando 25 kB de memoria.

**Rendimiento inicial:** El tiempo de ejecución registrado por `EXPLAIN ANALYZE` fue de 0.206 ms. La mediana de tres mediciones fue de **0.084 ms**, que se conservará como referencia para la comparación posterior a la creación de índices.

**Limitaciones:** La condición utiliza únicamente un límite superior de fecha, por lo que también podría incluir licencias vencidas. La consulta original se conserva sin modificaciones para respetar el ejercicio de análisis y optimización.

In [6]:
# Query 2 — SQL original del archivo 02_queries_analysis.sql
consultas[2] = "SELECT \n    first_name,\n    last_name,\n    license_number,\n    license_expiry\nFROM drivers\nWHERE license_expiry < CURRENT_DATE + INTERVAL '30 days'\nORDER BY license_expiry;"
print(consultas[2])


SELECT 
    first_name,
    last_name,
    license_number,
    license_expiry
FROM drivers
WHERE license_expiry < CURRENT_DATE + INTERVAL '30 days'
ORDER BY license_expiry;


In [7]:
# Ejecutar esta celda ANTES de crear los índices
resultados[2] = consultar(consultas[2])
planes_antes[2] = explicar(consultas[2])
tiempos_antes[2] = medir(consultas[2])
print("Tiempo base (mediana, ms):", tiempos_antes[2])


,first_name,last_name,license_number,license_expiry
0,Marina,Fernández,2801823908,2026-11-14
1,Edinson,Rubio,2536778950,2026-11-18
2,Tatiana,Cortés,1854316681,2026-11-21
3,Camilo,Álvarez,5789617427,2026-11-25
4,Lorena,Díaz,6169243284,2026-11-25
5,Jesús,Ríos,3446737272,2026-11-26
6,David,Espitia,1510289448,2026-12-03
7,Elizabeth,Bermúdez,6636925105,2026-12-08
8,Lorena,Ramírez,6923301988,2026-12-11


Filas devueltas: 9
Sort  (cost=12.12..12.14 rows=8 width=29) (actual time=0.167..0.168 rows=9.00 loops=1)
  Sort Key: license_expiry
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=5
  ->  Seq Scan on drivers  (cost=0.00..12.00 rows=8 width=29) (actual time=0.034..0.141 rows=9.00 loops=1)
        Filter: (license_expiry < (CURRENT_DATE + '30 days'::interval))
        Rows Removed by Filter: 391
        Buffers: shared hit=5
Planning Time: 0.116 ms
Execution Time: 0.206 ms
Tiempo base (mediana, ms): 0.084


### Query 3. Total de viajes por estado

**Problema de negocio:** Monitorear operaciones en curso.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Hallazgo de negocio:** La consulta identificó 100,000 viajes registrados en FleetLogix, de los cuales 99,928 (99.928%) se encuentran completados y 72 (0.072%) permanecen en progreso. Esto permite conocer el estado general de las operaciones de transporte y detectar viajes pendientes de finalizar.

**Plan de ejecución:** PostgreSQL utilizó `Seq Scan` para recorrer los 100,000 registros de la tabla `trips`. Posteriormente aplicó `HashAggregate` para agrupar los resultados por estado, obteniendo dos categorías. La operación de agrupación utilizó 32 kB de memoria.

**Rendimiento inicial:** El tiempo reportado por `EXPLAIN ANALYZE` fue de 36.438 ms, mientras que la mediana de tres mediciones fue de **33.362 ms**. Esta última se conservará como referencia para evaluar el rendimiento después de la optimización.

**Observaciones y limitaciones:** La consulta realiza una agregación sobre todos los viajes, sin filtros temporales. Aunque requiere una lectura completa de la tabla, esto puede ser apropiado para el volumen de datos y la operación solicitada. La creación de un índice no garantiza una mejora para esta consulta.

In [8]:
# Query 3 — SQL original del archivo 02_queries_analysis.sql
consultas[3] = 'SELECT \n    status,\n    COUNT(*) as total_viajes\nFROM trips\nGROUP BY status;'
print(consultas[3])


SELECT 
    status,
    COUNT(*) as total_viajes
FROM trips
GROUP BY status;


In [9]:
# Ejecutar esta celda ANTES de crear los índices
resultados[3] = consultar(consultas[3])
planes_antes[3] = explicar(consultas[3])
tiempos_antes[3] = medir(consultas[3])
print("Tiempo base (mediana, ms):", tiempos_antes[3])


,status,total_viajes
0,completed,99928
1,in_progress,72


Filas devueltas: 2
HashAggregate  (cost=2532.00..2532.02 rows=2 width=18) (actual time=36.383..36.385 rows=2.00 loops=1)
  Group Key: status
  Batches: 1  Memory Usage: 32kB
  Buffers: shared hit=1032
  ->  Seq Scan on trips  (cost=0.00..2032.00 rows=100000 width=10) (actual time=0.099..5.493 rows=100000.00 loops=1)
        Buffers: shared hit=1032
Planning Time: 0.181 ms
Execution Time: 36.438 ms
Tiempo base (mediana, ms): 33.362


## 4. Consultas intermedias (4–8)

### Query 4. Total de entregas por ciudad destino en los últimos 2 meses

**Problema de negocio:** Identificar demanda por ciudad para planificación de recursos.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Hallazgo de negocio:** La consulta permitió analizar la actividad logística de cinco ciudades de destino durante los últimos 60 días, considerando el número de viajes, la cantidad de entregas y el peso total transportado. Entre los resultados visibles destacan Cali, con 5,536 entregas; Barranquilla, con 5,240; y Medellín, con 5,124. Esta información ayuda a identificar la demanda logística por destino y puede utilizarse para planificar la asignación de vehículos y conductores.

**Plan de ejecución:** PostgreSQL utilizó un `Parallel Seq Scan` sobre la tabla `deliveries`, operaciones `Hash Join` para relacionar las tablas, `Sort` y `Gather Merge` para organizar los registros, y `GroupAggregate` para calcular los indicadores por ciudad. El plan empleó dos trabajadores paralelos.

**Rendimiento inicial:** `EXPLAIN ANALYZE` reportó un tiempo de ejecución de 128.982 ms. La mediana de tres mediciones fue de **185.584 ms**, valor que se conservará como referencia inicial para evaluar el efecto de los índices.

**Observaciones y limitaciones:** La consulta requiere procesar un volumen considerable de entregas y realizar varias operaciones de unión y agregación. El uso de paralelismo contribuye al procesamiento de los datos, aunque las lecturas de disco y la carga del sistema pueden producir variaciones en los tiempos. La consulta se mantiene sin modificaciones para conservar su comportamiento original.

In [10]:
# Query 4 — SQL original del archivo 02_queries_analysis.sql
consultas[4] = "SELECT \n    r.destination_city,\n    COUNT(DISTINCT t.trip_id) as total_viajes,\n    COUNT(d.delivery_id) as total_entregas,\n    SUM(d.package_weight_kg) as peso_total_kg\nFROM routes r\nINNER JOIN trips t ON r.route_id = t.route_id\nINNER JOIN deliveries d ON t.trip_id = d.trip_id\nWHERE t.departure_datetime >= CURRENT_DATE - INTERVAL '60 days'\nGROUP BY r.destination_city\nORDER BY total_entregas DESC;"
print(consultas[4])


SELECT 
    r.destination_city,
    COUNT(DISTINCT t.trip_id) as total_viajes,
    COUNT(d.delivery_id) as total_entregas,
    SUM(d.package_weight_kg) as peso_total_kg
FROM routes r
INNER JOIN trips t ON r.route_id = t.route_id
INNER JOIN deliveries d ON t.trip_id = d.trip_id
WHERE t.departure_datetime >= CURRENT_DATE - INTERVAL '60 days'
GROUP BY r.destination_city
ORDER BY total_entregas DESC;


In [11]:
# Ejecutar esta celda ANTES de crear los índices
resultados[4] = consultar(consultas[4])
planes_antes[4] = explicar(consultas[4])
tiempos_antes[4] = medir(consultas[4])
print("Tiempo base (mediana, ms):", tiempos_antes[4])


,destination_city,total_viajes,total_entregas,peso_total_kg
0,Bogotá,1694,6776,2782616.54
1,Cartagena,1532,6128,2614531.43
2,Cali,1384,5536,2355777.80
3,Barranquilla,1310,5240,2128841.04
4,Medellín,1281,5124,2087232.55


Filas devueltas: 5
Sort  (cost=15378.55..15378.84 rows=116 width=57) (actual time=124.448..128.683 rows=5.00 loops=1)
  Sort Key: (count(d.delivery_id)) DESC
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=10553 read=6761
  ->  GroupAggregate  (cost=11701.67..15374.57 rows=116 width=57) (actual time=113.866..128.668 rows=5.00 loops=1)
        Group Key: r.destination_city
        Buffers: shared hit=10553 read=6761
        ->  Gather Merge  (cost=11701.67..15082.81 rows=29031 width=23) (actual time=110.244..121.123 rows=28804.00 loops=1)
              Workers Planned: 2
              Workers Launched: 2
              Buffers: shared hit=10553 read=6761
              ->  Sort  (cost=10701.65..10731.89 rows=12096 width=23) (actual time=62.581..63.086 rows=9601.33 loops=3)
                    Sort Key: r.destination_city, t.trip_id
                    Sort Method: quicksort  Memory: 780kB
                    Buffers: shared hit=10553 read=6761
                    Worker 0:  S

### Query 5. Conductores activos con cantidad de viajes completados

**Problema de negocio:** Evaluar carga de trabajo por conductor.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Hallazgo de negocio:** La consulta identificó 375 conductores activos con viajes registrados. Los conductores con mayor cantidad de viajes completados fueron Fernando Restrepo (392), María Oviedo (388), Tatiana Pérez (387), Ana Gómez (382) y Paulo Mejía (382). Esta información permite evaluar la distribución de la carga de trabajo y reconocer a los conductores con mayor actividad operativa.

**Plan de ejecución:** PostgreSQL realizó recorridos secuenciales (`Seq Scan`) sobre las tablas `trips` y `drivers`. Posteriormente utilizó un `Hash Right Join` para relacionarlas, un `HashAggregate` para calcular los viajes totales y completados por conductor y una operación `Sort` para presentar los resultados en orden descendente.

**Rendimiento inicial:** El tiempo reportado por `EXPLAIN ANALYZE` fue de 83.448 ms. La mediana de tres mediciones fue de **75.16 ms**, que se utilizará como referencia para la comparación posterior a la creación de índices.

**Observaciones y limitaciones:** La consulta procesó los 100,000 registros de viajes y filtró los conductores activos. Aunque el rendimiento inicial es relativamente bajo en términos absolutos, el recorrido completo de la tabla `trips` representa un aspecto que puede analizarse durante la optimización. Los resultados describen la actividad acumulada en la base de datos, no necesariamente la productividad durante un periodo específico.

In [12]:
# Query 5 — SQL original del archivo 02_queries_analysis.sql
consultas[5] = "SELECT \n    d.driver_id,\n    d.first_name || ' ' || d.last_name as nombre_completo,\n    d.license_expiry,\n    COUNT(t.trip_id) as viajes_totales,\n    SUM(CASE WHEN t.status = 'completed' THEN 1 ELSE 0 END) as viajes_completados\nFROM drivers d\nLEFT JOIN trips t ON d.driver_id = t.driver_id\nWHERE d.status = 'active'\nGROUP BY d.driver_id, d.first_name, d.last_name, d.license_expiry\nHAVING COUNT(t.trip_id) > 0\nORDER BY viajes_completados DESC;"
print(consultas[5])


SELECT 
    d.driver_id,
    d.first_name || ' ' || d.last_name as nombre_completo,
    d.license_expiry,
    COUNT(t.trip_id) as viajes_totales,
    SUM(CASE WHEN t.status = 'completed' THEN 1 ELSE 0 END) as viajes_completados
FROM drivers d
LEFT JOIN trips t ON d.driver_id = t.driver_id
WHERE d.status = 'active'
GROUP BY d.driver_id, d.first_name, d.last_name, d.license_expiry
HAVING COUNT(t.trip_id) > 0
ORDER BY viajes_completados DESC;


In [13]:
# Ejecutar esta celda ANTES de crear los índices
resultados[5] = consultar(consultas[5])
planes_antes[5] = explicar(consultas[5])
tiempos_antes[5] = medir(consultas[5])
print("Tiempo base (mediana, ms):", tiempos_antes[5])


,driver_id,nombre_completo,license_expiry,viajes_totales,viajes_completados
0,275,Fernando Restrepo,2029-10-28,392,392
1,318,María Oviedo,2029-10-16,388,388
2,292,Tatiana Pérez,2029-08-26,387,387
3,203,Ana Gómez,2028-09-05,382,382
4,208,Paulo Mejía,2028-12-06,382,382
5,10,Mario Ariza,2029-02-14,381,381
6,180,Juan Puentes,2027-09-01,379,379
7,213,Marcos Parra,2029-01-11,378,378
8,350,Lucía García,2028-09-20,377,377
9,194,Daniela Caicedo,2027-07-05,375,375


Filas devueltas: 375
Sort  (cost=3259.13..3259.45 rows=125 width=70) (actual time=83.328..83.376 rows=375.00 loops=1)
  Sort Key: (sum(CASE WHEN ((t.status)::text = 'completed'::text) THEN 1 ELSE 0 END)) DESC
  Sort Method: quicksort  Memory: 52kB
  Buffers: shared hit=1037
  ->  HashAggregate  (cost=3249.47..3254.78 rows=125 width=70) (actual time=83.135..83.221 rows=375.00 loops=1)
        Group Key: d.driver_id
        Filter: (count(t.trip_id) > 0)
        Batches: 1  Memory Usage: 72kB
        Buffers: shared hit=1037
        ->  Hash Right Join  (cost=14.69..2311.97 rows=93750 width=36) (actual time=0.233..47.173 rows=100000.00 loops=1)
              Hash Cond: (t.driver_id = d.driver_id)
              Buffers: shared hit=1037
              ->  Seq Scan on trips t  (cost=0.00..2032.00 rows=100000 width=18) (actual time=0.008..6.191 rows=100000.00 loops=1)
                    Buffers: shared hit=1032
              ->  Hash  (cost=10.00..10.00 rows=375 width=22) (actual time=0.216.

### Query 6. Promedio de entregas por conductor en los últimos 6 meses

**Problema de negocio:** Medir productividad individual de conductores.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Hallazgo de negocio:** La consulta identificó 370 conductores que realizaron al menos 10 viajes completados durante los últimos seis meses. Permite analizar el volumen de entregas gestionadas por cada conductor y evaluar su productividad operativa.

Entre los resultados observados, Jorge Rodríguez registró 75 viajes y 300 entregas, mientras que Inés Galvis registró 47 viajes y 188 entregas. Ambos presentan un promedio de 4.00 entregas por viaje. El mismo promedio aparece en los demás resultados visibles, lo que limita la utilidad de esta métrica para diferenciar el desempeño individual.

**Plan de ejecución:** PostgreSQL realizó un recorrido secuencial de la tabla `deliveries`, que contiene 400,000 registros, y utilizó operaciones `Hash Join` para relacionar entregas, viajes y conductores. Posteriormente efectuó operaciones de ordenamiento y agregación para calcular las métricas solicitadas.

**Rendimiento inicial:** El tiempo de ejecución observado en `EXPLAIN ANALYZE` fue de 287.987 ms. La mediana de tres mediciones fue de **216.893 ms**, que se conservará como referencia para evaluar la optimización posterior.

**Observaciones y limitaciones:** El plan muestra un ordenamiento externo (`external merge`) que utilizó aproximadamente 3.6 MB de espacio temporal en disco. Este comportamiento representa una oportunidad de análisis del rendimiento. Además, los datos sintéticos presentan una distribución uniforme de cuatro entregas por viaje, por lo que esta métrica no permite identificar diferencias significativas entre los conductores observados. El promedio diario se calcula sobre 180 días fijos y no sobre días efectivamente trabajados.

In [14]:
# Query 6 — SQL original del archivo 02_queries_analysis.sql
consultas[6] = "SELECT \n    dr.driver_id,\n    dr.first_name || ' ' || dr.last_name as conductor,\n    COUNT(DISTINCT t.trip_id) as total_viajes,\n    COUNT(d.delivery_id) as total_entregas,\n    ROUND(COUNT(d.delivery_id)::NUMERIC / NULLIF(COUNT(DISTINCT t.trip_id), 0), 2) as promedio_entregas_por_viaje,\n    ROUND(COUNT(d.delivery_id)::NUMERIC / 180, 2) as promedio_entregas_diarias\nFROM drivers dr\nINNER JOIN trips t ON dr.driver_id = t.driver_id\nINNER JOIN deliveries d ON t.trip_id = d.trip_id\nWHERE t.departure_datetime >= CURRENT_DATE - INTERVAL '6 months'\n    AND t.status = 'completed'\nGROUP BY dr.driver_id, dr.first_name, dr.last_name\nHAVING COUNT(DISTINCT t.trip_id) >= 10\nORDER BY promedio_entregas_por_viaje DESC;"
print(consultas[6])


SELECT 
    dr.driver_id,
    dr.first_name || ' ' || dr.last_name as conductor,
    COUNT(DISTINCT t.trip_id) as total_viajes,
    COUNT(d.delivery_id) as total_entregas,
    ROUND(COUNT(d.delivery_id)::NUMERIC / NULLIF(COUNT(DISTINCT t.trip_id), 0), 2) as promedio_entregas_por_viaje,
    ROUND(COUNT(d.delivery_id)::NUMERIC / 180, 2) as promedio_entregas_diarias
FROM drivers dr
INNER JOIN trips t ON dr.driver_id = t.driver_id
INNER JOIN deliveries d ON t.trip_id = d.trip_id
WHERE t.departure_datetime >= CURRENT_DATE - INTERVAL '6 months'
    AND t.status = 'completed'
GROUP BY dr.driver_id, dr.first_name, dr.last_name
HAVING COUNT(DISTINCT t.trip_id) >= 10
ORDER BY promedio_entregas_por_viaje DESC;


In [15]:
# Ejecutar esta celda ANTES de crear los índices
resultados[6] = consultar(consultas[6])
planes_antes[6] = explicar(consultas[6])
tiempos_antes[6] = medir(consultas[6])
print("Tiempo base (mediana, ms):", tiempos_antes[6])


,driver_id,conductor,total_viajes,total_entregas,promedio_entregas_por_viaje,promedio_entregas_diarias
0,1,Jorge Rodríguez,75,300,4.00,1.67
1,2,Inés Galvis,47,188,4.00,1.04
2,3,Andrea Salazar,62,248,4.00,1.38
3,5,Alejandra Bernal,61,244,4.00,1.36
4,6,Marina Fernández,55,220,4.00,1.22
5,7,Jhon Herrera,62,248,4.00,1.38
6,8,René Quintero,68,272,4.00,1.51
7,10,Mario Ariza,90,360,4.00,2.00
8,12,Álvaro Beltrán,17,68,4.00,0.38
9,13,Liliana García,66,264,4.00,1.47


Filas devueltas: 370
Sort  (cost=25445.91..25446.25 rows=133 width=130) (actual time=279.825..279.838 rows=370.00 loops=1)
  Sort Key: (round(((count(d.delivery_id))::numeric / (NULLIF(count(DISTINCT t.trip_id), 0))::numeric), 2)) DESC
  Sort Method: quicksort  Memory: 56kB
  Buffers: shared hit=12888 read=5727, temp read=457 written=458
  ->  GroupAggregate  (cost=24453.12..25441.22 rows=133 width=130) (actual time=247.067..279.611 rows=370.00 loops=1)
        Group Key: dr.driver_id
        Filter: (count(DISTINCT t.trip_id) >= 10)
        Rows Removed by Filter: 5
        Buffers: shared hit=12888 read=5727, temp read=457 written=458
        ->  Sort  (cost=24453.12..24698.06 rows=97978 width=26) (actual time=246.882..258.685 rows=96460.00 loops=1)
              Sort Key: dr.driver_id, t.trip_id
              Sort Method: external merge  Disk: 3656kB
              Buffers: shared hit=12888 read=5727, temp read=457 written=458
              ->  Hash Join  (cost=1325.75..13985.66 rows

### Query 7. Rutas con mayor consumo de combustible por kilómetro

**Problema de negocio:** Identificar rutas ineficientes para optimización.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Hallazgo de negocio:** La consulta identifica las diez rutas con mayor consumo promedio de combustible por cada 100 kilómetros. Esta información permite detectar trayectos potencialmente menos eficientes y priorizar investigaciones sobre costos de operación, características de los vehículos y condiciones de las rutas.

Entre los resultados visibles, la ruta R011 (Bogotá → Cartagena) registró 2,031 viajes y un consumo promedio de 11.55 litros por cada 100 km. Por su parte, la ruta R017 (Medellín → Cali) registró 2,000 viajes y 11.54 litros por cada 100 km. Ambas aparecen en las posiciones 9 y 10 del ranking.

**Plan de ejecución:** PostgreSQL utilizó recorridos secuenciales (`Seq Scan`) sobre las tablas `trips` y `routes`, seguidos de un `Hash Join` para relacionar los registros. Posteriormente realizó una agregación (`HashAggregate`) por ruta y un ordenamiento `top-N heapsort` para seleccionar las diez rutas con mayor consumo normalizado.

**Rendimiento inicial:** El tiempo reportado por `EXPLAIN ANALYZE` fue de 124.252 ms. La mediana de tres mediciones fue de **100.566 ms**, valor que se conservará como referencia para la comparación posterior a la optimización.

**Observaciones y limitaciones:** La consulta procesó 99,928 viajes completados y devolvió diez rutas. El plan muestra un recorrido secuencial de la tabla de viajes, por lo que será relevante evaluar si los índices propuestos mejoran el rendimiento. Sin embargo, dado que la consulta necesita procesar prácticamente todos los viajes, PostgreSQL podría seguir prefiriendo un recorrido secuencial incluso después de crear índices.

El consumo de combustible por cada 100 kilómetros permite comparar rutas de diferentes distancias, aunque no controla diferencias en tipo de vehículo, carga transportada o condiciones de circulación.

In [16]:
# Query 7 — SQL original del archivo 02_queries_analysis.sql
consultas[7] = "SELECT \n    r.route_code,\n    r.origin_city || ' -> ' || r.destination_city as ruta,\n    r.distance_km,\n    COUNT(t.trip_id) as viajes_realizados,\n    AVG(t.fuel_consumed_liters) as promedio_combustible_litros,\n    ROUND(AVG(t.fuel_consumed_liters / NULLIF(r.distance_km, 0)) * 100, 2) as litros_por_100km,\n    SUM(t.fuel_consumed_liters) as combustible_total\nFROM routes r\nINNER JOIN trips t ON r.route_id = t.route_id\nWHERE t.fuel_consumed_liters IS NOT NULL \n    AND r.distance_km > 0\n    AND t.status = 'completed'\nGROUP BY r.route_id, r.route_code, r.origin_city, r.destination_city, r.distance_km\nHAVING COUNT(t.trip_id) >= 50\nORDER BY litros_por_100km DESC\nLIMIT 10;"
print(consultas[7])


SELECT 
    r.route_code,
    r.origin_city || ' -> ' || r.destination_city as ruta,
    r.distance_km,
    COUNT(t.trip_id) as viajes_realizados,
    AVG(t.fuel_consumed_liters) as promedio_combustible_litros,
    ROUND(AVG(t.fuel_consumed_liters / NULLIF(r.distance_km, 0)) * 100, 2) as litros_por_100km,
    SUM(t.fuel_consumed_liters) as combustible_total
FROM routes r
INNER JOIN trips t ON r.route_id = t.route_id
WHERE t.fuel_consumed_liters IS NOT NULL 
    AND r.distance_km > 0
    AND t.status = 'completed'
GROUP BY r.route_id, r.route_code, r.origin_city, r.destination_city, r.distance_km
HAVING COUNT(t.trip_id) >= 50
ORDER BY litros_por_100km DESC
LIMIT 10;


In [17]:
# Ejecutar esta celda ANTES de crear los índices
resultados[7] = consultar(consultas[7])
planes_antes[7] = explicar(consultas[7])
tiempos_antes[7] = medir(consultas[7])
print("Tiempo base (mediana, ms):", tiempos_antes[7])


,route_code,ruta,distance_km,viajes_realizados,promedio_combustible_litros,litros_por_100km,combustible_total
0,R022,Cali -> Bogotá,432.79,1955,50.1188439897698210,11.58,97982.34
1,R035,Barranquilla -> Medellín,621.55,1985,71.9321763224181360,11.57,142785.37
2,R008,Bogotá -> Barranquilla,1014.99,2051,117.3862554851292053,11.57,240759.21
3,R042,Cartagena -> Bogotá,1058.46,1990,122.2448241206030151,11.55,243267.20
4,R047,Cartagena -> Barranquilla,149.21,2071,17.2301641718976340,11.55,35683.67
5,R027,Cali -> Barranquilla,1076.70,1977,124.3880627212948912,11.55,245915.20
6,R033,Barranquilla -> Bogotá,1034.29,2011,119.4100149179512680,11.55,240133.54
7,R030,Cali -> Cartagena,1066.34,2017,123.1717600396628656,11.55,248437.44
8,R011,Bogotá -> Cartagena,1080.58,2031,124.8338060068931561,11.55,253537.46
9,R017,Medellín -> Cali,423.62,2000,48.8825550000000000,11.54,97765.11


Filas devueltas: 10
Limit  (cost=4116.14..4116.17 rows=10 width=170) (actual time=124.172..124.175 rows=10.00 loops=1)
  Buffers: shared hit=1033
  ->  Sort  (cost=4116.14..4117.07 rows=370 width=170) (actual time=124.171..124.173 rows=10.00 loops=1)
        Sort Key: (round((avg((t.fuel_consumed_liters / NULLIF(r.distance_km, '0'::numeric))) * '100'::numeric), 2)) DESC
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=1033
        ->  HashAggregate  (cost=4082.25..4108.15 rows=370 width=170) (actual time=124.075..124.142 rows=50.00 loops=1)
              Group Key: r.route_code
              Filter: (count(t.trip_id) >= 50)
              Batches: 1  Memory Usage: 121kB
              Buffers: shared hit=1033
              ->  Hash Join  (cost=37.75..2583.10 rows=99943 width=44) (actual time=0.052..54.676 rows=99928.00 loops=1)
                    Hash Cond: (t.route_id = r.route_id)
                    Buffers: shared hit=1033
                    ->  Seq Sca

### Query 8. Análisis de entregas retrasadas por día de la semana

**Problema de negocio:** Identificar patrones de retraso para mejorar planificación.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Hallazgo de negocio:** La consulta permite identificar los días de la semana con mayor proporción de entregas retrasadas durante el periodo analizado. Se considera retrasada una entrega realizada más de 30 minutos después de su horario programado.

El sábado presentó la mayor proporción de retrasos, con 687 entregas retrasadas de 6,552, equivalentes al 10.49%. Por el contrario, el jueves presentó el menor porcentaje, con 563 retrasos de 6,153 entregas, equivalentes al 9.15%.

Estos resultados permiten orientar el seguimiento de la puntualidad y evaluar posibles ajustes operativos en los días con mayor incidencia de retrasos.

**Plan de ejecución:** PostgreSQL utilizó un `Parallel Seq Scan` sobre la tabla `deliveries`, aplicando filtros por estado de entrega y fecha programada. Posteriormente realizó operaciones `Sort` y `Gather Merge` para organizar los registros, y finalmente un `GroupAggregate` para calcular los indicadores agrupados por día de la semana.

**Rendimiento inicial:** El tiempo de ejecución reportado por `EXPLAIN ANALYZE` fue de 206.318 ms. La mediana de tres mediciones fue de **194.179 ms**, que se utilizará como referencia antes de crear los índices de optimización.

**Observaciones y limitaciones:** La consulta utiliza funciones de fecha para agrupar las entregas y realiza un recorrido paralelo de la tabla. Será necesario evaluar si los índices propuestos reducen el costo de filtrado. Además, los resultados muestran diferencias descriptivas entre días, pero no permiten establecer por sí solos las causas de los retrasos.

In [18]:
# Query 8 — SQL original del archivo 02_queries_analysis.sql
consultas[8] = "SELECT \n    TO_CHAR(d.scheduled_datetime, 'Day') as dia_semana,\n    EXTRACT(DOW FROM d.scheduled_datetime) as num_dia,\n    COUNT(*) as total_entregas,\n    COUNT(CASE \n        WHEN d.delivered_datetime > d.scheduled_datetime + INTERVAL '30 minutes' \n        THEN 1 \n    END) as entregas_retrasadas,\n    ROUND(100.0 * COUNT(CASE \n        WHEN d.delivered_datetime > d.scheduled_datetime + INTERVAL '30 minutes' \n        THEN 1 \n    END) / COUNT(*), 2) as porcentaje_retrasos,\n    AVG(EXTRACT(EPOCH FROM (d.delivered_datetime - d.scheduled_datetime)) / 60) as minutos_promedio_diferencia\nFROM deliveries d\nWHERE d.delivery_status = 'delivered'\n    AND d.scheduled_datetime >= CURRENT_DATE - INTERVAL '90 days'\nGROUP BY dia_semana, num_dia\nORDER BY num_dia;"
print(consultas[8])


SELECT 
    TO_CHAR(d.scheduled_datetime, 'Day') as dia_semana,
    EXTRACT(DOW FROM d.scheduled_datetime) as num_dia,
    COUNT(*) as total_entregas,
    COUNT(CASE 
        WHEN d.delivered_datetime > d.scheduled_datetime + INTERVAL '30 minutes' 
        THEN 1 
    END) as entregas_retrasadas,
    ROUND(100.0 * COUNT(CASE 
        WHEN d.delivered_datetime > d.scheduled_datetime + INTERVAL '30 minutes' 
        THEN 1 
    END) / COUNT(*), 2) as porcentaje_retrasos,
    AVG(EXTRACT(EPOCH FROM (d.delivered_datetime - d.scheduled_datetime)) / 60) as minutos_promedio_diferencia
FROM deliveries d
WHERE d.delivery_status = 'delivered'
    AND d.scheduled_datetime >= CURRENT_DATE - INTERVAL '90 days'
GROUP BY dia_semana, num_dia
ORDER BY num_dia;


In [19]:
# Ejecutar esta celda ANTES de crear los índices
resultados[8] = consultar(consultas[8])
planes_antes[8] = explicar(consultas[8])
tiempos_antes[8] = medir(consultas[8])
print("Tiempo base (mediana, ms):", tiempos_antes[8])


,dia_semana,num_dia,total_entregas,entregas_retrasadas,porcentaje_retrasos,minutos_promedio_diferencia
0,Sunday,0,6677,637,9.54,11.49917231612750237135
1,Monday,1,6552,633,9.66,11.62715770384361009362
2,Tuesday,2,6587,656,9.96,12.05829122436870603715
3,Wednesday,3,6576,647,9.84,11.64811627934154501212
4,Thursday,4,6153,563,9.15,11.02362449124004550621
5,Friday,5,5954,566,9.51,11.22313881809987683349
6,Saturday,6,6552,687,10.49,12.81245766374898249895


Filas devueltas: 7
GroupAggregate  (cost=13066.66..20620.95 rows=44057 width=144) (actual time=126.674..206.025 rows=7.00 loops=1)
  Group Key: (EXTRACT(dow FROM scheduled_datetime)), (to_char(scheduled_datetime, 'Day'::text))
  Buffers: shared hit=2346 read=5069
  ->  Gather Merge  (cost=13066.66..18197.82 rows=44057 width=80) (actual time=119.039..169.484 rows=45051.00 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        Buffers: shared hit=2346 read=5069
        ->  Sort  (cost=12066.63..12112.52 rows=18357 width=80) (actual time=84.290..85.053 rows=15017.00 loops=3)
              Sort Key: (EXTRACT(dow FROM scheduled_datetime)), (to_char(scheduled_datetime, 'Day'::text))
              Sort Method: quicksort  Memory: 1655kB
              Buffers: shared hit=2346 read=5069
              Worker 0:  Sort Method: quicksort  Memory: 1042kB
              Worker 1:  Sort Method: quicksort  Memory: 952kB
              ->  Parallel Seq Scan on deliveries d  (cost=0.00..107

## 5. Consultas complejas (9–12)

### Query 9. Costo de mantenimiento por kilómetro recorrido

**Problema de negocio:** Evaluar costo-beneficio de cada tipo de vehículo.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Hallazgo de negocio:** La consulta busca determinar el costo de mantenimiento por kilómetro recorrido para cada tipo de vehículo, con el objetivo de evaluar la eficiencia económica de la flota y apoyar la planificación de gastos operativos.

**Plan de ejecución:** PostgreSQL utiliza operaciones `Hash Join`, `Sort`, `GroupAggregate` y `Subquery Scan`. Durante la ejecución se generan 3,307,460 registros intermedios. El método de ordenamiento `External Merge` requiere 185,440 kB de espacio temporal en disco, lo que incrementa el costo de procesamiento.

**Rendimiento inicial:** El tiempo reportado por `EXPLAIN ANALYZE` fue de 8,306.067 ms, mientras que la mediana de tres mediciones fue de 9,141.623 ms.

**Observaciones y limitaciones:** Se identificó un problema lógico en la consulta original debido a la unión simultánea de las tablas de viajes y mantenimientos por vehículo. Esta operación multiplica los registros antes de realizar las agregaciones, provocando una sobreestimación de los kilómetros recorridos y los costos acumulados de mantenimiento.

Se propone como mejora realizar las agregaciones de viajes y mantenimientos de forma independiente mediante CTEs, para posteriormente combinar sus resultados por vehículo. Se conserva la consulta original como referencia para el análisis de rendimiento.

In [20]:
# Query 9 — SQL original del archivo 02_queries_analysis.sql
consultas[9] = "WITH vehicle_metrics AS (\n    SELECT \n        v.vehicle_id,\n        v.vehicle_type,\n        v.license_plate,\n        COUNT(DISTINCT t.trip_id) as total_viajes,\n        SUM(r.distance_km) as km_totales,\n        SUM(m.cost) as costo_mantenimiento_total,\n        COUNT(DISTINCT m.maintenance_id) as cantidad_mantenimientos\n    FROM vehicles v\n    LEFT JOIN trips t ON v.vehicle_id = t.vehicle_id\n    LEFT JOIN routes r ON t.route_id = r.route_id\n    LEFT JOIN maintenance m ON v.vehicle_id = m.vehicle_id\n    WHERE t.status = 'completed'\n    GROUP BY v.vehicle_id, v.vehicle_type, v.license_plate\n)\nSELECT \n    vehicle_type,\n    COUNT(vehicle_id) as cantidad_vehiculos,\n    SUM(total_viajes) as viajes_totales,\n    SUM(km_totales) as kilometros_totales,\n    SUM(costo_mantenimiento_total) as costo_total_mantenimiento,\n    ROUND(SUM(costo_mantenimiento_total) / NULLIF(SUM(km_totales), 0), 2) as costo_por_km,\n    ROUND(AVG(costo_mantenimiento_total / NULLIF(cantidad_mantenimientos, 0)), 2) as costo_promedio_por_mantenimiento\nFROM vehicle_metrics\nWHERE km_totales > 0 AND costo_mantenimiento_total > 0\nGROUP BY vehicle_type\nORDER BY costo_por_km DESC;"
print(consultas[9])


WITH vehicle_metrics AS (
    SELECT 
        v.vehicle_id,
        v.vehicle_type,
        v.license_plate,
        COUNT(DISTINCT t.trip_id) as total_viajes,
        SUM(r.distance_km) as km_totales,
        SUM(m.cost) as costo_mantenimiento_total,
        COUNT(DISTINCT m.maintenance_id) as cantidad_mantenimientos
    FROM vehicles v
    LEFT JOIN trips t ON v.vehicle_id = t.vehicle_id
    LEFT JOIN routes r ON t.route_id = r.route_id
    LEFT JOIN maintenance m ON v.vehicle_id = m.vehicle_id
    WHERE t.status = 'completed'
    GROUP BY v.vehicle_id, v.vehicle_type, v.license_plate
)
SELECT 
    vehicle_type,
    COUNT(vehicle_id) as cantidad_vehiculos,
    SUM(total_viajes) as viajes_totales,
    SUM(km_totales) as kilometros_totales,
    SUM(costo_mantenimiento_total) as costo_total_mantenimiento,
    ROUND(SUM(costo_mantenimiento_total) / NULLIF(SUM(km_totales), 0), 2) as costo_por_km,
    ROUND(AVG(costo_mantenimiento_total / NULLIF(cantidad_mantenimientos, 0)), 2) as costo_pr

In [21]:
# Ejecutar esta celda ANTES de crear los índices
resultados[9] = consultar(consultas[9])
planes_antes[9] = explicar(consultas[9])
tiempos_antes[9] = medir(consultas[9])
print("Tiempo base (mediana, ms):", tiempos_antes[9])


,vehicle_type,cantidad_vehiculos,viajes_totales,kilometros_totales,costo_total_mantenimiento,costo_por_km,costo_promedio_por_mantenimiento
0,Camión Mediano,47,23835,549245922.71,241049844166.27,438.87,159038252.01
1,Camión Grande,50,29522,713574878.22,309913694395.67,434.31,184474049.99
2,Motocicleta,19,11253,269576597.30,116358987016.20,431.64,183593860.64
3,Van,62,35296,848130185.01,363814121446.38,428.96,175947415.02


Filas devueltas: 4
Sort  (cost=487877.59..487877.65 rows=22 width=179) (actual time=8222.789..8222.795 rows=4.00 loops=1)
  Sort Key: (round((sum(vehicle_metrics.costo_mantenimiento_total) / NULLIF(sum(vehicle_metrics.km_totales), '0'::numeric)), 2)) DESC
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=1122, temp read=46355 written=46428
  ->  GroupAggregate  (cost=487875.89..487877.10 rows=22 width=179) (actual time=8222.721..8222.788 rows=4.00 loops=1)
        Group Key: vehicle_metrics.vehicle_type
        Buffers: shared hit=1122, temp read=46355 written=46428
        ->  Sort  (cost=487875.89..487875.95 rows=22 width=95) (actual time=8222.691..8222.702 rows=178.00 loops=1)
              Sort Key: vehicle_metrics.vehicle_type
              Sort Method: quicksort  Memory: 37kB
              Buffers: shared hit=1122, temp read=46355 written=46428
              ->  Subquery Scan on vehicle_metrics  (cost=450392.56..487875.40 rows=22 width=95) (actual time=6148.794..8222.3

### Query 10. Ranking de conductores por eficiencia usando Window Functions

**Problema de negocio:** Identificar top performers para incentivos.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Hallazgo de negocio:** La consulta permite identificar a los 20 conductores con mejor desempeño operativo durante los últimos tres meses, considerando la puntualidad, la eficiencia en el consumo de combustible y la cantidad de entregas realizadas. Se evaluaron 362 conductores que cumplieron con el requisito mínimo de 20 viajes.

**Plan de ejecución:** PostgreSQL utiliza operaciones `WindowAgg`, `Sort` y `Limit`. Las funciones de ventana permiten establecer clasificaciones individuales mediante `RANK()`, mientras que el método `Top-N Heapsort` facilita la selección de los 20 mejores resultados.

**Rendimiento inicial:** El tiempo reportado por `EXPLAIN ANALYZE` fue de 207.357 ms, mientras que el tiempo base mediano fue de 200.698 ms.

**Observaciones y limitaciones:** La consulta asigna el mismo peso a los tres indicadores de desempeño. No considera las diferencias entre tipos de vehículos y puede penalizar las entregas pendientes al calcular la puntualidad. Asimismo, la unión con la tabla de entregas puede influir en el promedio de consumo de combustible cuando los viajes tienen diferentes cantidades de entregas asociadas. Al tratarse de datos sintéticos, los resultados deben interpretarse como una simulación del desempeño operativo.

In [22]:
# Query 10 — SQL original del archivo 02_queries_analysis.sql
consultas[10] = "WITH conductor_metricas AS (\n    SELECT \n        d.driver_id,\n        d.first_name || ' ' || d.last_name as nombre,\n        COUNT(DISTINCT t.trip_id) as viajes,\n        COUNT(DISTINCT del.delivery_id) as entregas,\n        AVG(t.fuel_consumed_liters / NULLIF(r.distance_km, 0)) * 100 as consumo_100km,\n        COUNT(CASE WHEN del.delivered_datetime <= del.scheduled_datetime THEN 1 END)::NUMERIC / \n            NULLIF(COUNT(del.delivery_id), 0) * 100 as puntualidad_pct\n    FROM drivers d\n    JOIN trips t ON d.driver_id = t.driver_id\n    JOIN routes r ON t.route_id = r.route_id\n    LEFT JOIN deliveries del ON t.trip_id = del.trip_id\n    WHERE t.departure_datetime >= CURRENT_DATE - INTERVAL '3 months'\n    GROUP BY d.driver_id, d.first_name, d.last_name\n    HAVING COUNT(DISTINCT t.trip_id) >= 20\n)\nSELECT \n    nombre,\n    viajes,\n    entregas,\n    ROUND(consumo_100km, 2) as consumo_100km,\n    ROUND(puntualidad_pct, 2) as puntualidad_pct,\n    RANK() OVER (ORDER BY puntualidad_pct DESC) as rank_puntualidad,\n    RANK() OVER (ORDER BY consumo_100km ASC) as rank_eficiencia,\n    RANK() OVER (ORDER BY entregas DESC) as rank_productividad,\n    (RANK() OVER (ORDER BY puntualidad_pct DESC) + \n     RANK() OVER (ORDER BY consumo_100km ASC) + \n     RANK() OVER (ORDER BY entregas DESC)) / 3.0 as score_promedio\nFROM conductor_metricas\nORDER BY score_promedio ASC\nLIMIT 20;"
print(consultas[10])


WITH conductor_metricas AS (
    SELECT 
        d.driver_id,
        d.first_name || ' ' || d.last_name as nombre,
        COUNT(DISTINCT t.trip_id) as viajes,
        COUNT(DISTINCT del.delivery_id) as entregas,
        AVG(t.fuel_consumed_liters / NULLIF(r.distance_km, 0)) * 100 as consumo_100km,
        COUNT(CASE WHEN del.delivered_datetime <= del.scheduled_datetime THEN 1 END)::NUMERIC / 
            NULLIF(COUNT(del.delivery_id), 0) * 100 as puntualidad_pct
    FROM drivers d
    JOIN trips t ON d.driver_id = t.driver_id
    JOIN routes r ON t.route_id = r.route_id
    LEFT JOIN deliveries del ON t.trip_id = del.trip_id
    WHERE t.departure_datetime >= CURRENT_DATE - INTERVAL '3 months'
    GROUP BY d.driver_id, d.first_name, d.last_name
    HAVING COUNT(DISTINCT t.trip_id) >= 20
)
SELECT 
    nombre,
    viajes,
    entregas,
    ROUND(consumo_100km, 2) as consumo_100km,
    ROUND(puntualidad_pct, 2) as puntualidad_pct,
    RANK() OVER (ORDER BY puntualidad_pct DESC) as rank_p

In [23]:
# Ejecutar esta celda ANTES de crear los índices
resultados[10] = consultar(consultas[10])
planes_antes[10] = explicar(consultas[10])
tiempos_antes[10] = medir(consultas[10])
print("Tiempo base (mediana, ms):", tiempos_antes[10])


,nombre,viajes,entregas,consumo_100km,puntualidad_pct,rank_puntualidad,rank_eficiencia,rank_productividad,score_promedio
0,Erick Martínez,38,152,10.87,54.61,12,18,45,25.0000000000000000
1,Esteban Valencia,40,160,11.09,50.00,55,41,24,40.0000000000000000
2,Leonor Saavedra,34,136,10.84,56.62,5,16,107,42.6666666666666667
3,Patricia López,41,164,11.30,54.27,16,110,20,48.6666666666666667
4,Leonel Pardo,39,156,11.26,51.92,37,90,32,53.0000000000000000
5,Carmen Jaimes,40,160,11.19,49.38,68,68,24,53.3333333333333333
6,Ramiro Guerrero,45,180,10.99,46.67,134,27,5,55.3333333333333333
7,Hernán Garzón,37,148,11.18,49.32,70,62,59,63.6666666666666667
8,Carlos Oviedo,32,128,11.04,54.69,10,33,156,66.3333333333333333
9,Andrea Contreras,35,140,11.01,48.57,89,32,86,69.0000000000000000


Filas devueltas: 20
Limit  (cost=18038.05..18038.10 rows=20 width=232) (actual time=200.325..200.331 rows=20.00 loops=1)
  Buffers: shared hit=8765 read=3941, temp read=416 written=417
  ->  Sort  (cost=18038.05..18038.38 rows=133 width=232) (actual time=200.325..200.329 rows=20.00 loops=1)
        Sort Key: (((((rank() OVER w3 + (rank() OVER w2)) + (rank() OVER w1)))::numeric / 3.0))
        Sort Method: top-N heapsort  Memory: 32kB
        Buffers: shared hit=8765 read=3941, temp read=416 written=417
        ->  WindowAgg  (cost=18030.21..18034.51 rows=133 width=232) (actual time=199.859..200.195 rows=362.00 loops=1)
              Window: w3 AS (ORDER BY conductor_metricas.puntualidad_pct ROWS UNBOUNDED PRECEDING)
              Storage: Memory  Maximum Storage: 17kB
              Buffers: shared hit=8765 read=3941, temp read=416 written=417
              ->  Sort  (cost=18030.19..18030.52 rows=133 width=128) (actual time=199.853..199.869 rows=362.00 loops=1)
                    Sort 

### Query 11. Análisis de tendencia de viajes con LAG y LEAD

**Problema de negocio:** Proyectar necesidades futuras basadas en tendencias.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Hallazgo de negocio:** La consulta permite analizar la evolución mensual de los viajes completados, identificar variaciones en la actividad logística y evaluar tendencias mediante un promedio móvil de tres meses. Se analizaron 99,928 viajes completados y se mostraron los 12 periodos mensuales más recientes.

**Operadores del plan relevantes:** `Seq Scan`, `Sort`, `GroupAggregate`, `WindowAgg` y `Limit`. PostgreSQL emplea funciones de ventana para comparar periodos consecutivos y calcular promedios móviles.

**Rendimiento inicial:** El tiempo de ejecución reportado por `EXPLAIN ANALYZE` fue de 142.923 ms, mientras que el tiempo base mediano fue de 123.374 ms.

**Observaciones y limitaciones:** El plan de ejecución utiliza almacenamiento temporal en disco durante el ordenamiento de los registros. Las comparaciones mensuales pueden verse afectadas por la diferente duración de los meses y por periodos con información incompleta. Asimismo, los datos sintéticos no necesariamente reflejan tendencias estacionales reales. La función `LEAD()` permite consultar valores posteriores existentes, pero no constituye una predicción de viajes futuros.

In [24]:
# Query 11 — SQL original del archivo 02_queries_analysis.sql
consultas[11] = "WITH viajes_mensuales AS (\n    SELECT \n        DATE_TRUNC('month', departure_datetime) as mes,\n        COUNT(*) as total_viajes,\n        SUM(total_weight_kg) as peso_total,\n        AVG(fuel_consumed_liters) as combustible_promedio\n    FROM trips\n    WHERE status = 'completed'\n    GROUP BY DATE_TRUNC('month', departure_datetime)\n)\nSELECT \n    TO_CHAR(mes, 'YYYY-MM') as periodo,\n    total_viajes,\n    LAG(total_viajes, 1) OVER (ORDER BY mes) as viajes_mes_anterior,\n    LEAD(total_viajes, 1) OVER (ORDER BY mes) as viajes_mes_siguiente,\n    total_viajes - LAG(total_viajes, 1) OVER (ORDER BY mes) as cambio_absoluto,\n    ROUND((total_viajes - LAG(total_viajes, 1) OVER (ORDER BY mes))::NUMERIC / \n          NULLIF(LAG(total_viajes, 1) OVER (ORDER BY mes), 0) * 100, 2) as cambio_porcentual,\n    ROUND(peso_total / 1000, 2) as toneladas_transportadas,\n    ROUND(combustible_promedio, 2) as combustible_promedio_viaje,\n    AVG(total_viajes) OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as promedio_movil_3m\nFROM viajes_mensuales\nORDER BY mes DESC\nLIMIT 12;"
print(consultas[11])


WITH viajes_mensuales AS (
    SELECT 
        DATE_TRUNC('month', departure_datetime) as mes,
        COUNT(*) as total_viajes,
        SUM(total_weight_kg) as peso_total,
        AVG(fuel_consumed_liters) as combustible_promedio
    FROM trips
    WHERE status = 'completed'
    GROUP BY DATE_TRUNC('month', departure_datetime)
)
SELECT 
    TO_CHAR(mes, 'YYYY-MM') as periodo,
    total_viajes,
    LAG(total_viajes, 1) OVER (ORDER BY mes) as viajes_mes_anterior,
    LEAD(total_viajes, 1) OVER (ORDER BY mes) as viajes_mes_siguiente,
    total_viajes - LAG(total_viajes, 1) OVER (ORDER BY mes) as cambio_absoluto,
    ROUND((total_viajes - LAG(total_viajes, 1) OVER (ORDER BY mes))::NUMERIC / 
          NULLIF(LAG(total_viajes, 1) OVER (ORDER BY mes), 0) * 100, 2) as cambio_porcentual,
    ROUND(peso_total / 1000, 2) as toneladas_transportadas,
    ROUND(combustible_promedio, 2) as combustible_promedio_viaje,
    AVG(total_viajes) OVER (ORDER BY mes ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)

In [25]:
# Ejecutar esta celda ANTES de crear los índices
resultados[11] = consultar(consultas[11])
planes_antes[11] = explicar(consultas[11])
tiempos_antes[11] = medir(consultas[11])
print("Tiempo base (mediana, ms):", tiempos_antes[11])


,periodo,total_viajes,viajes_mes_anterior,viajes_mes_siguiente,cambio_absoluto,cambio_porcentual,toneladas_transportadas,combustible_promedio_viaje,promedio_movil_3m
0,2026-11,553,4247,NaN,-3694,-86.98,1020.00,81.01,2970.0000000000000000
1,2026-10,4247,4110,553.0,137,3.33,7389.66,83.69,4201.0000000000000000
2,2026-09,4110,4246,4247.0,-136,-3.20,7164.13,82.12,4201.0000000000000000
3,2026-08,4246,4247,4110.0,-1,-0.02,7569.49,83.13,4200.6666666666666667
4,2026-07,4247,4109,4246.0,138,3.36,7446.58,84.31,4201.0000000000000000
5,2026-06,4109,4247,4247.0,-138,-3.25,7142.42,81.94,4155.3333333333333333
6,2026-05,4247,4110,4109.0,137,3.33,7604.28,82.16,4201.0000000000000000
7,2026-04,4110,4246,4247.0,-136,-3.20,7157.85,83.22,4064.0000000000000000
8,2026-03,4246,3836,4110.0,410,10.69,7518.57,82.56,4109.3333333333333333
9,2026-02,3836,4246,4246.0,-410,-9.66,6646.79,82.39,4109.6666666666666667


Filas devueltas: 12
Limit  (cost=24418.43..24418.46 rows=12 width=200) (actual time=135.198..135.201 rows=12.00 loops=1)
  Buffers: shared hit=1032, temp read=392 written=393
  ->  Sort  (cost=24418.43..24668.29 rows=99943 width=200) (actual time=135.197..135.198 rows=12.00 loops=1)
        Sort Key: (date_trunc('month'::text, trips.departure_datetime)) DESC
        Sort Method: top-N heapsort  Memory: 28kB
        Buffers: shared hit=1032, temp read=392 written=393
        ->  WindowAgg  (cost=12882.64..22127.26 rows=99943 width=200) (actual time=89.941..135.170 rows=25.00 loops=1)
              Window: w2 AS (ORDER BY (date_trunc('month'::text, trips.departure_datetime)) ROWS BETWEEN '2'::bigint PRECEDING AND CURRENT ROW)
              Storage: Memory  Maximum Storage: 17kB
              Buffers: shared hit=1032, temp read=392 written=393
              ->  WindowAgg  (cost=12882.58..17629.83 rows=99943 width=96) (actual time=87.762..134.871 rows=25.00 loops=1)
                    Win

### Query 12. Pivot de entregas por hora y día de la semana

**Problema de negocio:** Optimizar horarios de operación y personal.

**Procedimiento:** ejecutar el SQL original, observar sus resultados y registrar el plan y el tiempo inicial **antes de crear índices**. Si no devuelve filas, analizar las condiciones y la cobertura temporal; no modificar la consulta original.

**Hallazgo de negocio:** La consulta permite identificar patrones de demanda logística mediante la distribución de las entregas programadas por hora y día de la semana durante los últimos 60 días. Se analizaron 22,577 registros, distribuidos en 17 franjas horarias entre las 06:00 y las 22:59. Los resultados pueden utilizarse para optimizar la asignación de vehículos, conductores y recursos operativos.

**Operadores del plan relevantes:** `Parallel Seq Scan`, `Sort`, `Gather Merge`, `GroupAggregate` y `Subquery Scan`. PostgreSQL utiliza procesamiento paralelo con dos trabajadores adicionales para realizar la lectura y agrupación de los registros.

**Rendimiento inicial:** El tiempo de ejecución reportado por `EXPLAIN ANALYZE` fue de 188.270 ms, mientras que el tiempo base mediano fue de 214.496 ms.

**Observaciones y limitaciones:** La consulta contabiliza las entregas según su fecha y hora programadas, independientemente de su estado. Excluye las entregas fuera de las franjas horarias seleccionadas. La columna `total_semana` representa el acumulado por hora durante todo el periodo analizado y no el total de una semana individual. Debido a la naturaleza sintética de los datos, los patrones encontrados no necesariamente representan una demanda logística real.

In [26]:
# Query 12 — SQL original del archivo 02_queries_analysis.sql
consultas[12] = "WITH entregas_por_hora_dia AS (\n    SELECT \n        EXTRACT(DOW FROM scheduled_datetime) as dia_semana,\n        EXTRACT(HOUR FROM scheduled_datetime) as hora,\n        COUNT(*) as cantidad_entregas\n    FROM deliveries\n    WHERE scheduled_datetime >= CURRENT_DATE - INTERVAL '60 days'\n    GROUP BY EXTRACT(DOW FROM scheduled_datetime), EXTRACT(HOUR FROM scheduled_datetime)\n)\nSELECT \n    hora,\n    SUM(CASE WHEN dia_semana = 0 THEN cantidad_entregas ELSE 0 END) as domingo,\n    SUM(CASE WHEN dia_semana = 1 THEN cantidad_entregas ELSE 0 END) as lunes,\n    SUM(CASE WHEN dia_semana = 2 THEN cantidad_entregas ELSE 0 END) as martes,\n    SUM(CASE WHEN dia_semana = 3 THEN cantidad_entregas ELSE 0 END) as miercoles,\n    SUM(CASE WHEN dia_semana = 4 THEN cantidad_entregas ELSE 0 END) as jueves,\n    SUM(CASE WHEN dia_semana = 5 THEN cantidad_entregas ELSE 0 END) as viernes,\n    SUM(CASE WHEN dia_semana = 6 THEN cantidad_entregas ELSE 0 END) as sabado,\n    SUM(cantidad_entregas) as total_semana\nFROM entregas_por_hora_dia\nWHERE hora BETWEEN 6 AND 22\nGROUP BY hora\nORDER BY hora;"
print(consultas[12])


WITH entregas_por_hora_dia AS (
    SELECT 
        EXTRACT(DOW FROM scheduled_datetime) as dia_semana,
        EXTRACT(HOUR FROM scheduled_datetime) as hora,
        COUNT(*) as cantidad_entregas
    FROM deliveries
    WHERE scheduled_datetime >= CURRENT_DATE - INTERVAL '60 days'
    GROUP BY EXTRACT(DOW FROM scheduled_datetime), EXTRACT(HOUR FROM scheduled_datetime)
)
SELECT 
    hora,
    SUM(CASE WHEN dia_semana = 0 THEN cantidad_entregas ELSE 0 END) as domingo,
    SUM(CASE WHEN dia_semana = 1 THEN cantidad_entregas ELSE 0 END) as lunes,
    SUM(CASE WHEN dia_semana = 2 THEN cantidad_entregas ELSE 0 END) as martes,
    SUM(CASE WHEN dia_semana = 3 THEN cantidad_entregas ELSE 0 END) as miercoles,
    SUM(CASE WHEN dia_semana = 4 THEN cantidad_entregas ELSE 0 END) as jueves,
    SUM(CASE WHEN dia_semana = 5 THEN cantidad_entregas ELSE 0 END) as viernes,
    SUM(CASE WHEN dia_semana = 6 THEN cantidad_entregas ELSE 0 END) as sabado,
    SUM(cantidad_entregas) as total_semana
FROM ent

In [27]:
# Ejecutar esta celda ANTES de crear los índices
resultados[12] = consultar(consultas[12])
planes_antes[12] = explicar(consultas[12])
tiempos_antes[12] = medir(consultas[12])
print("Tiempo base (mediana, ms):", tiempos_antes[12])


,hora,domingo,lunes,martes,miercoles,jueves,viernes,sabado,total_semana
0,6,83,106,108,113,94,83,88,675
1,7,116,128,126,107,126,119,107,829
2,8,133,140,167,147,130,117,114,948
3,9,145,147,160,180,166,139,134,1071
4,10,171,166,191,161,194,149,173,1205
5,11,176,195,190,194,223,181,161,1320
6,12,180,224,211,223,212,185,185,1420
7,13,191,195,206,228,240,180,183,1423
8,14,195,238,249,236,209,203,176,1506
9,15,200,244,237,234,223,225,193,1556


Filas devueltas: 17
GroupAggregate  (cost=12958.45..12968.46 rows=138 width=288) (actual time=169.550..187.880 rows=17.00 loops=1)
  Group Key: entregas_por_hora_dia.hora
  Buffers: shared hit=4116 read=3283
  ->  Sort  (cost=12958.45..12958.80 rows=138 width=72) (actual time=169.518..187.716 rows=119.00 loops=1)
        Sort Key: entregas_por_hora_dia.hora
        Sort Method: quicksort  Memory: 29kB
        Buffers: shared hit=4116 read=3283
        ->  Subquery Scan on entregas_por_hora_dia  (cost=12932.99..12953.55 rows=138 width=72) (actual time=157.612..187.628 rows=119.00 loops=1)
              Buffers: shared hit=4116 read=3283
              ->  GroupAggregate  (cost=12932.99..12952.17 rows=138 width=72) (actual time=157.611..187.615 rows=119.00 loops=1)
                    Group Key: (EXTRACT(dow FROM deliveries.scheduled_datetime)), (EXTRACT(hour FROM deliveries.scheduled_datetime))
                    Buffers: shared hit=4116 read=3283
                    ->  Gather Merge  (

## 6. Resumen de tiempos de la línea base

Ejecuta esta sección **solo después de haber medido las 12 consultas**. Guarda el archivo notebook con sus salidas para conservar evidencia.

In [28]:
tabla_base = pd.DataFrame([
    {"query": n, "nivel": "Básica" if n <= 3 else "Intermedia" if n <= 8 else "Compleja",
     "tiempo_antes_ms": tiempos_antes.get(n)}
    for n in range(1, 13)
])
display(tabla_base)
print(f"Mediciones registradas: {len(tiempos_antes)}/12")


,query,nivel,tiempo_antes_ms
0,1,Básica,0.134
1,2,Básica,0.084
2,3,Básica,33.362
3,4,Intermedia,185.584
4,5,Intermedia,75.160
5,6,Intermedia,216.893
6,7,Intermedia,100.566
7,8,Intermedia,194.179
8,9,Compleja,9141.623
9,10,Compleja,200.698


Mediciones registradas: 12/12


## 7. Propuesta de cinco índices — después de revisar los planes

**No hay índices nuevos aplicados por este notebook.** Identifica los filtros, las uniones y los ordenamientos costosos observados en `planes_antes` y propón cinco índices concretos. Comprueba previamente los índices existentes en PostgreSQL para no duplicarlos. Las claves primarias ya suelen tener índices.

El siguiente bloque **solo genera un borrador SQL**. Reemplaza cada marcador por una instrucción `CREATE INDEX` justificada, revísala y ejecútala **manualmente en DBeaver**. No se recomienda crear índices a ciegas: algunos pueden no aportar beneficios y aumentan el costo de escritura.

In [30]:
sql_indices_existentes = """
SELECT tablename, indexname, indexdef
FROM pg_indexes
WHERE schemaname = 'public'
  AND tablename IN ('vehicles','drivers','routes','trips','deliveries','maintenance')
ORDER BY tablename, indexname;
"""
indices_existentes = consultar(sql_indices_existentes, limite_visualizacion=100)

print(indices_existentes.to_string(index=False))

,tablename,indexname,indexdef
0,deliveries,deliveries_pkey,CREATE UNIQUE INDEX deliveries_pkey ON public....
1,deliveries,deliveries_tracking_number_key,CREATE UNIQUE INDEX deliveries_tracking_number...
2,deliveries,idx_deliveries_status,CREATE INDEX idx_deliveries_status ON public.d...
3,drivers,drivers_employee_code_key,CREATE UNIQUE INDEX drivers_employee_code_key ...
4,drivers,drivers_license_number_key,CREATE UNIQUE INDEX drivers_license_number_key...
5,drivers,drivers_pkey,CREATE UNIQUE INDEX drivers_pkey ON public.dri...
6,maintenance,maintenance_pkey,CREATE UNIQUE INDEX maintenance_pkey ON public...
7,routes,routes_pkey,CREATE UNIQUE INDEX routes_pkey ON public.rout...
8,routes,routes_route_code_key,CREATE UNIQUE INDEX routes_route_code_key ON p...
9,trips,idx_trips_departure,CREATE INDEX idx_trips_departure ON public.tri...


Filas devueltas: 14
  tablename                      indexname                                                                                              indexdef
 deliveries                deliveries_pkey                    CREATE UNIQUE INDEX deliveries_pkey ON public.deliveries USING btree (delivery_id)
 deliveries deliveries_tracking_number_key CREATE UNIQUE INDEX deliveries_tracking_number_key ON public.deliveries USING btree (tracking_number)
 deliveries          idx_deliveries_status                 CREATE INDEX idx_deliveries_status ON public.deliveries USING btree (delivery_status)
    drivers      drivers_employee_code_key           CREATE UNIQUE INDEX drivers_employee_code_key ON public.drivers USING btree (employee_code)
    drivers     drivers_license_number_key         CREATE UNIQUE INDEX drivers_license_number_key ON public.drivers USING btree (license_number)
    drivers                   drivers_pkey                            CREATE UNIQUE INDEX drivers_pkey ON publ

In [31]:

# Propuesta de cinco índices basada en los planes de ejecución

propuesta_indices = [
    {
        "indice": 1,
        "tabla_columnas": "deliveries (trip_id)",
        "query_beneficiada": "4, 6, 10",
        "justificacion": "Facilitar las uniones entre entregas y viajes."
    },
    {
        "indice": 2,
        "tabla_columnas": "deliveries (scheduled_datetime)",
        "query_beneficiada": "8, 12",
        "justificacion": "Optimizar los filtros por intervalos de fechas."
    },
    {
        "indice": 3,
        "tabla_columnas": "trips (driver_id)",
        "query_beneficiada": "5, 6, 10",
        "justificacion": "Facilitar las uniones entre viajes y conductores."
    },
    {
        "indice": 4,
        "tabla_columnas": "trips (vehicle_id)",
        "query_beneficiada": "9",
        "justificacion": "Facilitar la búsqueda de viajes asociados a cada vehículo."
    },
    {
        "indice": 5,
        "tabla_columnas": "maintenance (vehicle_id)",
        "query_beneficiada": "9",
        "justificacion": "Facilitar la búsqueda de mantenimientos por vehículo."
    }
]

display(pd.DataFrame(propuesta_indices))

print("Escribe y ejecuta los cinco CREATE INDEX manualmente en DBeaver, tras su revisión.")


,indice,tabla_columnas,query_beneficiada,justificacion
0,1,deliveries (trip_id),"4, 6, 10",Facilitar las uniones entre entregas y viajes.
1,2,deliveries (scheduled_datetime),"8, 12",Optimizar los filtros por intervalos de fechas.
2,3,trips (driver_id),"5, 6, 10",Facilitar las uniones entre viajes y conductores.
3,4,trips (vehicle_id),9,Facilitar la búsqueda de viajes asociados a ca...
4,5,maintenance (vehicle_id),9,Facilitar la búsqueda de mantenimientos por ve...


Escribe y ejecuta los cinco CREATE INDEX manualmente en DBeaver, tras su revisión.


## 8. Segunda medición: después de crear los cinco índices

**Ejecutar únicamente cuando los índices ya existan en la base de datos.** El código mantiene la misma SQL, las mismas repeticiones y la misma conexión para facilitar la comparación. Las mejoras no están garantizadas: PostgreSQL puede seguir eligiendo un escaneo secuencial si es más eficiente.

In [32]:
# Cambia a True SOLO cuando hayas creado y verificado los cinco índices en DBeaver.
indices_creados_y_verificados = True
if not indices_creados_y_verificados:
    print("Pendiente: crear y verificar los cinco índices antes de medir el después.")
else:
    for n in range(1, 13):
        planes_despues[n] = explicar(consultas[n])
        tiempos_despues[n] = medir(consultas[n])
    print("Segunda medición terminada.")


Sort  (cost=5.08..5.09 rows=4 width=19) (actual time=0.255..0.256 rows=4.00 loops=1)
  Sort Key: (count(*)) DESC
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=2
  ->  HashAggregate  (cost=5.00..5.04 rows=4 width=19) (actual time=0.209..0.210 rows=4.00 loops=1)
        Group Key: vehicle_type
        Batches: 1  Memory Usage: 32kB
        Buffers: shared hit=2
        ->  Seq Scan on vehicles  (cost=0.00..4.00 rows=200 width=11) (actual time=0.066..0.093 rows=200.00 loops=1)
              Buffers: shared hit=2
Planning Time: 0.306 ms
Execution Time: 0.400 ms
Sort  (cost=12.12..12.14 rows=8 width=29) (actual time=0.100..0.101 rows=9.00 loops=1)
  Sort Key: license_expiry
  Sort Method: quicksort  Memory: 25kB
  Buffers: shared hit=5
  ->  Seq Scan on drivers  (cost=0.00..12.00 rows=8 width=29) (actual time=0.017..0.093 rows=9.00 loops=1)
        Filter: (license_expiry < (CURRENT_DATE + '30 days'::interval))
        Rows Removed by Filter: 391
        Buffers: shared hit=5

## 9. Comparación de rendimiento

La mejora porcentual se calcula como `(tiempo_antes - tiempo_despues) / tiempo_antes × 100`. Un valor negativo indica que la consulta fue más lenta. Compara además los planes y la cantidad de buffers leídos; no atribuyas todos los cambios a los índices sin considerar caché, estadísticas y variabilidad.

In [33]:
comparacion = pd.DataFrame([
    {"query": n, "nivel": "Básica" if n <= 3 else "Intermedia" if n <= 8 else "Compleja",
     "antes_ms": tiempos_antes.get(n), "despues_ms": tiempos_despues.get(n)}
    for n in range(1, 13)
])
comparacion["mejora_pct"] = (
    (comparacion["antes_ms"] - comparacion["despues_ms"])
    / comparacion["antes_ms"].replace(0, float("nan")) * 100
).round(2)
display(comparacion)


,query,nivel,antes_ms,despues_ms,mejora_pct
0,1,Básica,0.134,0.106,20.90
1,2,Básica,0.084,0.099,-17.86
2,3,Básica,33.362,37.400,-12.10
3,4,Intermedia,185.584,138.333,25.46
4,5,Intermedia,75.160,86.234,-14.73
5,6,Intermedia,216.893,215.359,0.71
6,7,Intermedia,100.566,99.721,0.84
7,8,Intermedia,194.179,89.882,53.71
8,9,Compleja,9141.623,2080.104,77.25
9,10,Compleja,200.698,184.376,8.13


In [34]:
# Validación de resultados de las 12 consultas

validacion_resultados = pd.DataFrame([
    {
        "query": n,
        "filas_devueltas": len(resultados[n]),
        "estado": "Sin resultados" if resultados[n].empty
        else "Con resultados"
    }
    for n in range(1, 13)
])

display(validacion_resultados)

,query,filas_devueltas,estado
0,1,4,Con resultados
1,2,9,Con resultados
2,3,2,Con resultados
3,4,5,Con resultados
4,5,375,Con resultados
5,6,370,Con resultados
6,7,10,Con resultados
7,8,7,Con resultados
8,9,4,Con resultados
9,10,20,Con resultados


In [35]:
# Validación de los rangos de fechas

sql_rango_fechas = """
SELECT
    'trips' AS tabla,
    MIN(departure_datetime) AS fecha_minima,
    MAX(departure_datetime) AS fecha_maxima
FROM trips

UNION ALL

SELECT
    'deliveries' AS tabla,
    MIN(scheduled_datetime),
    MAX(scheduled_datetime)
FROM deliveries;
"""

consultar(sql_rango_fechas)

,tabla,fecha_minima,fecha_maxima
0,trips,2024-11-05 01:22:03.178014,2026-11-05 23:11:14.218014
1,deliveries,2024-11-05 02:38:21.498618,2026-11-06 00:56:14.218014


Filas devueltas: 2


,tabla,fecha_minima,fecha_maxima
0,trips,2024-11-05 01:22:03.178014,2026-11-05 23:11:14.218014
1,deliveries,2024-11-05 02:38:21.498618,2026-11-06 00:56:14.218014



## 10. Hallazgos, anomalías y limitaciones

### 10.1. Validación de los resultados

Se ejecutaron correctamente las 12 consultas SQL, obteniendo resultados en todos los casos. Ninguna consulta devolvió un conjunto de datos vacío.

Se verificaron los rangos temporales de las tablas principales:

- Viajes: del 5 de noviembre de 2024 al 5 de noviembre de 2026.
- Entregas programadas: del 5 de noviembre de 2024 al 6 de noviembre de 2026.

La base de datos contiene aproximadamente dos años de información histórica, lo que permite realizar análisis de tendencias y comparaciones temporales.

### 10.2. Anomalía identificada en la consulta 9

Se identificó un problema lógico en la consulta de costo de mantenimiento por kilómetro recorrido.

Al realizar un JOIN entre los viajes y los mantenimientos asociados a cada vehículo, se genera una multiplicación de filas que provoca una sobreestimación de los kilómetros recorridos y de los costos acumulados de mantenimiento.

Como propuesta de mejora, se recomienda realizar las agregaciones de viajes y mantenimientos por separado mediante CTE, para posteriormente unir los resultados utilizando vehicle_id.

Se conservó la consulta original para garantizar que las mediciones de rendimiento antes y después de crear los índices fueran comparables.

### 10.3. Diferencias en los criterios de puntualidad

Se identificaron diferentes criterios para evaluar la puntualidad de las entregas:

- Consulta 8: considera retrasada una entrega completada cuando supera los 30 minutos respecto al horario programado.
- Consulta 10: considera puntual una entrega cuando se realiza antes o exactamente en el horario programado, sin tolerancia adicional.

Además, la consulta 10 incluye en su denominador las entregas asociadas a los viajes analizados, sin exigir que estén completadas.

Se recomienda estandarizar estos criterios para garantizar la consistencia de los indicadores de desempeño.

### 10.4. Resultados de la optimización mediante índices

Se crearon cinco índices adicionales sobre las tablas deliveries, trips y maintenance, aumentando de 14 a 19 el número de índices existentes en las seis tablas analizadas.

Después de realizar una segunda medición de rendimiento, se identificaron los siguientes resultados:

- 9 de las 12 consultas redujeron su tiempo de ejecución.
- 3 consultas presentaron un incremento en su tiempo de ejecución.
- La consulta 12 obtuvo una mejora del 81.56%.
- La consulta 9 obtuvo una mejora del 77.25%.
- La consulta 8 obtuvo una mejora del 53.71%.

La consulta 9 presentó la mayor reducción absoluta, pasando de 9141.623 ms a 2080.104 ms.

Las consultas 2, 3 y 5 presentaron incrementos en sus tiempos de ejecución, mientras que las consultas 6 y 7 mostraron mejoras inferiores al 1%.

Los índices sobre fechas y claves foráneas pueden facilitar las operaciones de filtrado y unión. Sin embargo, PostgreSQL puede seguir utilizando escaneos secuenciales cuando estima que son más eficientes.

Para determinar la contribución específica de cada índice, es necesario contrastar los planes de ejecución completos antes y después de su creación.

### 10.5. Limitaciones del análisis

- Los datos utilizados son sintéticos y no necesariamente representan el comportamiento de una empresa logística real.
- Los tiempos de ejecución pueden variar debido al uso de memoria caché y a los recursos disponibles.
- El tamaño de las tablas influye en la elección de los planes de ejecución.
- Los filtros basados en CURRENT_DATE generan resultados variables según la fecha de ejecución.
- La creación de índices implica un costo adicional de almacenamiento y mantenimiento durante las operaciones de escritura.
- La optimización del rendimiento no garantiza la corrección lógica de las consultas.

### 10.6. Conclusión general

La implementación de índices permitió observar una mejora en el rendimiento de la mayoría de las consultas analizadas, especialmente en aquellas con operaciones complejas y filtros temporales.

El análisis también permitió identificar inconsistencias en los cálculos y diferencias en las definiciones de los indicadores de desempeño.

Se concluye que la optimización de una base de datos debe considerar tanto la eficiencia de las consultas como la calidad y consistencia de los resultados obtenidos.



## 11. Evidencias para la entrega

| Evidencia | Estado |
|---|---|
| Archivo oficial `02_queries_analysis.sql` | Incluido como fuente de las consultas |
| 12 resultados de consultas | Completado: todas devolvieron resultados |
| 12 planes `EXPLAIN (ANALYZE, BUFFERS)` iniciales | Ejecutados |
| Cinco índices creados y justificados | Completado y verificado en DBeaver |
| 12 planes de ejecución posteriores | Ejecutados |
| Tabla de tiempos y mejoras | Completado |
| Validación de rangos temporales | Completado |
| Conclusiones de negocio y técnicas | Documentadas |

**Conclusión:** Se ejecutaron y analizaron las 12 consultas SQL del proyecto FleetLogix. Se implementaron cinco índices adicionales y se comparó el rendimiento antes y después de su creación.

Los resultados mostraron una reducción en los tiempos de ejecución de nueve consultas, alcanzando mejoras de hasta el 81.56%.

También se documentaron posibles inconsistencias lógicas, diferencias en los criterios de puntualidad y limitaciones del análisis.


In [36]:
# Cerrar la conexión cuando hayas terminado de trabajar.
# Ejecuta esta celda AL FINAL; para seguir trabajando después, vuelve a ejecutar la conexión.
if not conn.closed:
    conn.close()
    print("Conexión cerrada correctamente.")


Conexión cerrada correctamente.
